# Protein folding as a combinatorial optimization problem (HP lattice model)

A toy model of a protein is a chain of **H** (hydrophobic) and **P** (polar) residues on a 2D square lattice.

- The chain must be connected (consecutive residues occupy adjacent lattice sites).
- No two residues may occupy the same site.
- Non-consecutive H residues that are lattice neighbors contribute a favorable contact.

We use **simulated annealing** to search for a conformation with many H–H contacts. This is an illustrative model—not a realistic protein-structure predictor.


In [ ]:
import random
import math
import matplotlib.pyplot as plt

random.seed(7)


## 1. Define a sequence and a valid initial chain

The sequence below is deliberately small so that the search and visualization are easy to follow. The initial chain is a self-avoiding random walk.


In [ ]:
sequence = "PHHPHHPPHHPH"  # H = hydrophobic, P = polar
N = len(sequence)

directions = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def random_walk(n, max_attempts=10_000):
    for _ in range(max_attempts):
        coords = [(0, 0)]
        occupied = {coords[0]}
        for _ in range(n - 1):
            options = [(coords[-1][0] + dx, coords[-1][1] + dy)
                       for dx, dy in directions]
            options = [p for p in options if p not in occupied]
            if not options:
                break
            p = random.choice(options)
            coords.append(p)
            occupied.add(p)
        if len(coords) == n:
            return coords
    raise RuntimeError("Could not generate a walk; try a shorter sequence.")

initial = random_walk(N)


## 2. Objective function: count hydrophobic contacts

A contact is a pair of H residues that are adjacent on the lattice, but are **not** neighbors along the chain itself. We maximize the number of such contacts.


In [ ]:
def contact_count(coords, seq):
    contacts = 0
    n = len(seq)
    for i in range(n):
        if seq[i] != "H":
            continue
        for j in range(i + 2, n):  # exclude chain-neighbor pairs
            if seq[j] != "H":
                continue
            dx = abs(coords[i][0] - coords[j][0])
            dy = abs(coords[i][1] - coords[j][1])
            if dx + dy == 1:
                contacts += 1
    return contacts

def plot_chain(coords, seq, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    xs, ys = zip(*coords)
    ax.plot(xs, ys, "-k", linewidth=1.5, zorder=1)
    for i, (x, y) in enumerate(coords):
        ax.scatter(x, y, s=350, c=("tab:red" if seq[i] == "H" else "tab:blue"),
                   edgecolor="black", zorder=2)
        ax.text(x, y, f"{i}\n{seq[i]}", ha="center", va="center",
                color="white", fontsize=9, zorder=3)
    ax.set_aspect("equal")
    ax.grid(True)
    ax.set_title(title)
    plt.show()

plot_chain(initial, sequence, f"Initial conformation: {contact_count(initial, sequence)} contacts")


## 3. Simulated annealing

We propose a move by rotating the chain tail around a randomly chosen interior pivot by 90°, 180°, or 270°. We reject moves that cause lattice overlap.

Simulated annealing accepts improvements, and sometimes accepts worse moves—with a probability that decreases as the temperature cools—helping it escape local optima.


In [ ]:
def rotate_tail(coords, pivot_idx, angle):
    px, py = coords[pivot_idx]
    tail = coords[pivot_idx + 1:]
    rotated = []
    for x, y in tail:
        dx, dy = x - px, y - py
        if angle == 90:
            rx, ry = -dy, dx
        elif angle == 180:
            rx, ry = -dx, -dy
        else:  # 270 degrees
            rx, ry = dy, -dx
        rotated.append((px + rx, py + ry))
    return coords[:pivot_idx + 1] + rotated

def is_valid(coords):
    return len(set(coords)) == len(coords)

def simulated_annealing(seq, steps=20_000, T0=2.0, cooling=0.9995):
    current = random_walk(len(seq))
    current_score = contact_count(current, seq)
    best, best_score = current[:], current_score
    temperature = T0

    scores = []
    for _ in range(steps):
        pivot = random.randrange(0, len(seq) - 2)
        angle = random.choice([90, 180, 270])
        proposal = rotate_tail(current, pivot, angle)

        if is_valid(proposal):
            score = contact_count(proposal, seq)
            delta = score - current_score
            if delta >= 0 or random.random() < math.exp(delta / max(temperature, 1e-12)):
                current, current_score = proposal, score
                if score > best_score:
                    best, best_score = current[:], score
        scores.append(best_score)
        temperature *= cooling

    return best, best_score, scores

best, best_score, history = simulated_annealing(sequence)
print("Sequence:", sequence)
print("Initial contacts:", contact_count(initial, sequence))
print("Best contacts found:", best_score)


In [ ]:
plot_chain(best, sequence, f"Best conformation found: {best_score} H–H contacts")

plt.figure(figsize=(8, 3))
plt.plot(history)
plt.xlabel("Iteration")
plt.ylabel("Best contact count so far")
plt.title("Simulated annealing search")
plt.grid(True)
plt.show()


## 4. Interpretation and experiments

- The decision variables are the lattice coordinates of residues (subject to chain and non-overlap constraints).
- The objective is to maximize favorable H–H contacts.
- The constraints make this a combinatorial search over valid conformations.
- Simulated annealing is a heuristic: it may find a good conformation, but it does **not** certify global optimality.

Try changing the sequence, number of iterations, initial temperature, or cooling rate. Because the method is stochastic, different runs can produce different results.
